## 👥 Feature Importance by Customer Demographics

### Why analyze by demographics?
Global feature importance tells us what drives churn **on average** across all customers. But churn drivers can vary significantly across different customer segments:
* A feature important for **new customers** may be irrelevant for long-term customers
* **Senior citizens** may churn for different reasons than younger customers
* **High-value customers** may respond to different retention strategies than low value ones

By analyzing feature importance across segments, we can help business design **targeted retention strategies** for each group rather than a one-size-fits-all approach.

### Segments we analyze:
1. **Tenure Group** - New vs. Developing vs. Established vs. Loyal Customers
2. **Senior Citizens** vs Non-Senior Citizens
3. **Contract Type** - Month-to-Month vs Long-term
4. **Customer Value** - High value vs low value

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

# Load processed data
df = pd.read_csv('../data/processed/telco_churn_processed.csv')

# Load best model — use XGBoost
with open('../models/xgboost.pkl', 'rb') as f:
    best_xgb = pickle.load(f)

# Recreate train/test split with same random state
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Generate predictions
y_pred_xgb = best_xgb.predict(X_test)
y_prob_xgb = best_xgb.predict_proba(X_test)[:, 1]

# Build analysis dataframe
test_df = X_test.copy()
test_df['Churn'] = y_test.values
test_df['Churn_Prob'] = y_prob_xgb

print(f'✅ Data loaded: {df.shape[0]} rows, {df.shape[1]} columns')
print(f'✅ Model loaded: XGBoost')
print(f'✅ Test set ready: {test_df.shape[0]} customers')
print(f'Churn rate in test set: {test_df["Churn"].mean():.2%}')

✅ Data loaded: 7043 rows, 34 columns
✅ Model loaded: XGBoost
✅ Test set ready: 1409 customers
Churn rate in test set: 26.54%


### 1. Churn Rate by Tenure Group
Tenure is one of the strongest churn predictors. Here we break down churn rate across our engineered tenure groups to visualize how customer loyalty evolves over time.

**Business Insight:** Understanding at which tenure stage customers are most at risk allows the business to time retention interventions more effectively - for example, targeting customers approaching the 12-month mark before they decide to leave.